# Fog-density detection

Compare YOLO11n, YOLO11m and DETR with RGB versus predicted fog density.

## 1. Configure

Set `ACTION` and experiment controls. Keep the output path to resume.

In [1]:
from pathlib import Path
import json
import os
import sys
import importlib
import pandas as pd
from IPython.display import display, Image


def configure_environment():
    root = Path.cwd().resolve()
    if root.name == 'notebooks':
        root = root.parent
    support = str(root / 'notebooks/_support')
    if support not in sys.path:
        sys.path.insert(0, support)
    os.environ.setdefault('YOLO_CONFIG_DIR', str(root / '.ultralytics'))
    os.environ.setdefault('HF_HOME', str(root / '.cache/huggingface'))
    os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
    return root


ROOT = configure_environment()
from fog_data import cityscapes_manifest, resolve_foggy_images
from fog_analytics import export_analytics
from fog_prepare import prepare_nuscenes
from fog_research import read_manifest
import fog_training
fog_training = importlib.reload(fog_training)
from fog_training import default_config, preflight, run_training


def configure_run(root):
    config = default_config(root)
    config.update(epochs=30, seeds=[42, 43, 44], size=640, accumulate=4,
    return config


ACTION = 'train'  # train, preflight, analyze
CONFIG = configure_run(ROOT)
CITY_ROOT = ROOT / 'data/fog/cityscapes'
DOWNLOAD_FOG_IMAGES = True
NUSCENES_ROOT = ROOT / 'data/depth/datasets/nuscenes'
assert ACTION in {'train', 'preflight', 'analyze'}
display(pd.DataFrame([{'setting': key, 'value': str(value)}
                      for key, value in CONFIG.items()]))

,setting,value
0,output,C:\Users\tsao1\OneDrive\Documents\workspace\Ai...
1,manifest,C:\Users\tsao1\OneDrive\Documents\workspace\Ai...
2,city_manifest,C:\Users\tsao1\OneDrive\Documents\workspace\Ai...
3,density_checkpoint,None
4,checkpoints,{'YOLO11n': 'C:\\Users\\tsao1\\OneDrive\\Docum...
5,arms,"['rgb_control', 'predicted_density']"
6,seeds,"[42, 43, 44]"
7,device,cuda:0
8,size,640
9,epochs,30


## 2. Prepare data

Build or reuse the complete nuScenes and Foggy Cityscapes manifests.

In [2]:
def prepare_manifests(root, config):
    if ACTION == 'analyze':
        return
    if not Path(config['manifest']).exists():
        prepare_nuscenes(NUSCENES_ROOT, config['manifest'])
    if not Path(config['city_manifest']).exists():
        source = resolve_foggy_images(root, CITY_ROOT, download=DOWNLOAD_FOG_IMAGES)
        if source is not None:
            cityscapes_manifest(source, config['city_manifest'])
        else:
            print('Foggy Cityscapes not found; density training is blocked.')


def load_rows(config):
    rows = []
    for key in ('manifest', 'city_manifest'):
        path = Path(config[key])
        if path.exists():
            print(f'Validating {path.name}')
            rows.extend(read_manifest(path))
    return rows


def show_data_summary(rows, config):
    if not rows:
        return
    frame = pd.DataFrame(rows)
    display(frame.groupby(['dataset', 'split']).agg(
        images=('id', 'size'), scenes=('group', 'nunique'), known_beta=('beta', 'count')))
    coverage_path = Path(config['manifest']).with_suffix('.coverage.json')
    if coverage_path.exists():
        coverage = json.loads(coverage_path.read_text())
        display(pd.DataFrame([{
            'expected': coverage['expected_images'],
            'available': coverage['available_images'],
            'missing': len(coverage['missing_images']),
            'complete': coverage['complete']}]))


prepare_manifests(ROOT, CONFIG)
rows = load_rows(CONFIG)
show_data_summary(rows, CONFIG)

Validating nuscenes_all.json
Validating cityscapes.json


images  scenes  known_beta
dataset          split                                     
foggy_cityscapes density_train     6816    1527        6816
                 test              1500     235        1500
                 val               2109     358        2109
nuscenes         detector_train  150948     626           0
                 test             36114     150           0
                 val              17832      74           0

,expected,available,missing,complete
0,204894,204894,0,True


## 3. Validate

Preflight stops before training when data, models or settings are invalid.

In [3]:
def validate_run(config, rows):
    checks = preflight(config, rows)
    display(checks)
    if not checks.ready.all():
        display(checks.loc[~checks.ready, ['check', 'detail']])
    return checks


checks = None if ACTION == 'analyze' else validate_run(CONFIG, rows)

,check,ready,detail
0,training images,True,150948 labeled images
1,tuning images,True,17832 scene-held-out images for checkpoint sel...
2,test images,True,36114 held-out images for final metrics
3,arms,True,"rgb_control, predicted_density"
4,scene/ID isolation,True,"No scene crosses train, tuning, density-traini..."
5,size,True,640
6,epochs,True,30
7,density_epochs,True,15
8,accumulate,True,4
9,save_every,True,200


## 4. Train and evaluate

Interrupt the kernel to checkpoint; rerun with the same output path to resume.

In [4]:
def execute_analysis(config):
    return export_analytics(config['output'])


def execute_training(action, config, rows, checks):
    if action == 'preflight':
        return pd.DataFrame(), None
    if not checks.ready.all():
        print('Training is blocked by the checks above.')
        return pd.DataFrame(), None
    return run_training(rows, config)


def execute_run(action, config, rows, checks):
    if action == 'analyze':
        return execute_analysis(config), None
    return execute_training(action, config, rows, checks)


results, status = execute_run(ACTION, CONFIG, rows, checks)
if status is not None:
    display(status)

Migrating compatible checkpoint metadata to the current resumable engine.


YOLO11n/rgb_control seed=42 epoch=1:  91%|######### | 137208/150948 [00:00<?, ?it/s]

YOLO11n/rgb_control tuning:   0%|          | 0/17832 [00:00<?, ?it/s]

YOLO11n/rgb_control seed=42 epoch=2:   0%|          | 0/150948 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Review

Reports include accuracy, latency, density error, paired deltas and saved checkpoints.

In [ ]:
def show_metrics(results, output):
    if results.empty:
        print('No completed runs yet.')
        return
    display(results[results.density == 'all'].round(4))
    paired = pd.read_csv(Path(output) / 'paired-deltas.csv', dtype={'density': str})
    display(paired.loc[paired.density == 'all',
                       ['model', 'dataset', 'seed', 'arm', 'delta_mAP', 'delta_latency_ms']].round(4))
    density = pd.read_csv(Path(output) / 'density-quality.csv')
    if not density.empty:
        display(density.round(6))


def show_figures(output, results):
    if results.empty:
        return
    for pattern in ('*-comparison.png', '*-deltas.png', '*-classes.png', '*-density.png', '*-loss.png'):
        for path in sorted((Path(output) / 'figures').glob(pattern)):
            display(Image(filename=str(path), width=1000))


show_metrics(results, CONFIG['output'])
show_figures(CONFIG['output'], results)

## Notes

nuScenes boxes are projected 2D proxies. Foggy Cityscapes beta comes from filenames. Results are saved under the configured output path.